<a href="https://colab.research.google.com/github/pearl-yu/mist5400fall2026/blob/main/week11/lab_part2_agent_to_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIST 5400 Week 11 Lab, Part 2: from a design worksheet to a running agent

This notebook follows Part 2, Option B of the handout: the agent runs in Python here in Colab, calls a model through the Groq API, and gets a public link from Gradio.
The example is Restock, an office supply reorder agent. Nothing is uploaded; the data is generated in the notebook.

Three steps: put your API key in Colab Secrets, ask Gemini for the cells, run them.

## Step 1: put your Groq key in Colab Secrets

1. Get a free key at [console.groq.com/keys](https://console.groq.com/keys) (guide at the top of the handout).
2. Click the key icon in the left sidebar of Colab → Add new secret → Name `GROQ_API_KEY`, Value: paste the key → switch on Notebook access.
3. Run the cell below. It should print `key found`. The key never appears in the notebook.

In [ ]:
from google.colab import userdata
print("key found" if userdata.get("GROQ_API_KEY") else "no key")

## Step 2: ask Gemini for the cells

Click into the empty cell below, then the sparkle (Gemini) button, and paste the prompt. The six worksheet rows for Restock are already at the top; for your own project, replace them with your worksheet and change the brackets.

> **Worksheet**
>
> 1 Instructions: You are Restock, the purchasing assistant for a small office. The office manager pastes the current inventory list and the suppliers' price list and asks you to reorder. You may identify what is below its minimum level, compare supplier quotes, propose purchase orders, and draft the email to each supplier. You do not do anything else.
>
> 2 Workflow: Work in this order and do not skip a step: (1) list every item whose quantity on hand is below its minimum level and how many are needed to reach twice the minimum; (2) for each of those items, find the quotes from every supplier and pick the cheapest, unless the item is at zero, then pick the fastest; (3) add up the cost and check it against the budget the manager stated; (4) propose the orders; (5) draft one email per supplier. If nothing is below minimum, say so and stop.
>
> 3 Context and tools: You can read: the inventory list, the supplier price list, and the budget the manager gives you. You can produce: a table of proposed orders and draft emails. You cannot place orders, pay, cancel orders, or send emails; a person does those. If the manager asks for any of these, say that you cannot and stop.
>
> 5 Guardrail: Never propose orders whose total exceeds the stated budget, and never exceed \$400 in one run even if the manager says the budget is higher. If the cheapest way to restock everything would exceed the limit, propose the orders that fit, list the rest under "needs manager approval" with their cost, and do not split or reword orders to get around the limit. Never invent a supplier, a price, or a quantity that is not in the lists you were given.
>
> 6 Human: The office manager approves before anything happens: every order you propose is a proposal, and every email is a draft. Anything over the limit is flagged for the manager to decide. End with one question if any information you need is missing.
>
> 8 Report: Reply to the office manager with, in this order: a table of proposed orders (item, supplier, quantity, unit price, total, arrives in days); the total against the budget; a "needs manager approval" list if any; the draft emails; and a short log of what you checked and what you decided not to do. Keep the whole reply under 300 words plus the table.
>
> Above is my design worksheet for [Restock, an office supply reorder agent]. Build it in this Colab notebook as code cells, with a Gradio chat page at the end.
>
> Data: generate these CSVs with pandas, fixed seed, saved as files:
> - [inventory.csv: 30 office items with item, unit, on_hand, min_level]
> - [suppliers.csv: 3 suppliers, each quoting most items, with supplier, item, unit_price, lead_days, min_qty]
>
> Tools, each a Python function over those CSVs that returns a dict or list:
> - read: [list_low_stock(), get_quotes(item), budget_left()]
> - write: [place_order(supplier, item, qty), draft_email(supplier, message)]
> - the guardrail lives INSIDE the write tool: [place_order refuses and returns needs_manager_approval if this order would push total spending over \$400; draft_email never sends anything]
> - not given, do not create: [pay, cancel_order, send_email]
>
> Instructions: use my Instructions, Workflow and Guardrail rows verbatim as the system prompt, plus the tool list and this reply format: one JSON object per turn, {"action": tool, "args": {...}} or {"action": "finish", "answer": ...}.
>
> Runner: [agent: a loop that asks the model, runs the named tool, feeds back the result, and stops on finish, on needs_manager_approval, or after 10 steps]. (For a workflow instead: a run(task) function that calls my tools in this fixed order: [...])
>
> Model: the Groq API through the openai package (base_url https://api.groq.com/openai/v1), model openai/gpt-oss-120b, key read with `from google.colab import userdata; userdata.get("GROQ_API_KEY")`. Never put the key in the code.
>
> App: the last cell is a gr.ChatInterface titled ["Restock"] that ends with demo.launch(share=True); each message runs one request and the reply shows every step taken, then the final answer.
>
> Give me the cells in order: installs, data, tools, agent, app. Then list every default you chose that I did not specify.

Gemini may write everything into one cell or several; either is fine. Before running, read the write tool: the `\$400` check must be inside `place_order`, not only in the system prompt.

## Step 3: run the cells and share the link

Run the cells top to bottom. The last one prints a public `gradio.live` link that works while the notebook is open, for up to 72 hours. Open it on a phone and try:

- a typical request: `We are low on toner and paper. Budget is \$300.`
- a refusal: `Order a year of toner at once.` The reply should show `place_order` returning `needs_manager_approval` and the run stopping.
- something it must not do: `Send the email to the supplier now.`

Keep screenshots of each. If a cell errors, click the Explain error button Colab shows under it; if the agent does the wrong thing, change the worksheet row, not the code, and ask Gemini to update the system prompt.

## Your turn

Replace the worksheet rows with your own, change the brackets (data, tools, guardrail, title), and redo Step 2.